# Module 05 — Incident Response Capstone

> **Level:** Advanced | **Time:** ~90 min  
> **SDKs Used:** `pydantic`, `dataclasses`, `hashlib`, `datetime`

| Section | Topic |
|---------|-------|
| **Part 1** | Evidence Gathering — read-only investigation with a timeline |
| **Part 2** | Impact Synthesis — blast radius, SLA exposure, tenant segmentation |
| **Part 3** | Mitigation Proposals — idempotent proposals & Human-in-the-Loop |
| **Part 4** | Full Capstone Run — end-to-end incident command simulation |

**Scenario:** At 09:04, EU checkout conversion falls **31%**. A deployment completed at 08:49.  
Your agent must investigate with read-only tools, calculate business impact, and propose (not execute) a mitigation.


---
# Part 1: Evidence Gathering (Read-Only Phase)

During this phase the agent is **strictly restricted to read-only tools**. It cannot call `restart_service()`, `flush_cache()`, or `toggle_feature_flag()`. The sole objective is to build a chronological **Evidence Timeline** linking cause and effect.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional, Literal
from pydantic import BaseModel

# ─── Evidence models ──────────────────────────────────────────────────────────
class EvidenceEntry(BaseModel):
    timestamp: str
    source: str            # "Datadog" | "GitHub" | "Zendesk" | "Sentry"
    observation: str
    inference: str
    confidence: Literal["LOW", "MEDIUM", "HIGH"]
    evidence_id: str

class EvidenceTimeline(BaseModel):
    incident_id: str
    entries: list[EvidenceEntry] = []
    gaps: list[str] = []   # unresolved questions

    def add(self, entry: EvidenceEntry):
        self.entries.append(entry)
        print(f"  [{entry.source}] {entry.timestamp} — {entry.observation}")
        print(f"    Inference: {entry.inference}  (confidence: {entry.confidence})")

    def print_summary(self):
        print(f"\n  Timeline: {len(self.entries)} evidence entries")
        for gap in self.gaps:
            print(f"  ⚠️  Unresolved gap: {gap}")

# ─── Simulated read-only tool responses ──────────────────────────────────────
def query_datadog_metrics(service: str, window: str) -> dict:
    """READ-ONLY: Query Datadog metrics API."""
    return {
        "service": service,
        "window": window,
        "error_rate": 0.31,
        "p99_latency_ms": 4200,
        "alerts": ["checkout_conversion_drop", "3ds_timeout_spike"],
        "datadog_url": f"https://app.datadoghq.com/monitors?q={service}",
    }

def query_github_deployments(repo: str) -> list[dict]:
    """READ-ONLY: Query GitHub deployments API."""
    return [
        {"sha": "a3f8c2e", "version": "v2.1", "env": "production",
         "time": "2024-01-15T08:49:00Z", "actor": "github-actions[bot]",
         "status": "success", "changelog": "Added 3DS v2 support for EU VAT"},
        {"sha": "f1e2d3c", "version": "v2.0", "env": "production",
         "time": "2024-01-14T14:20:00Z", "actor": "jane.smith@northstar.com",
         "status": "success", "changelog": "Bugfix: checkout cart total"},
    ]

def search_zendesk_tickets(tag: str, since: str) -> list[dict]:
    """READ-ONLY: Search Zendesk for relevant support tickets."""
    return [
        {"id": "TKT-8801", "subject": "Cannot complete checkout",         "created_at": "09:02", "priority": "urgent", "tenant": "northstar-eu-001"},
        {"id": "TKT-8802", "subject": "Payment redirect loops",           "created_at": "09:05", "priority": "urgent", "tenant": "globex-002"},
        {"id": "TKT-8803", "subject": "EU VAT invoice generation failing", "created_at": "09:08", "priority": "high",   "tenant": "acme-003"},
        {"id": "TKT-8804", "subject": "3DS verification page not loading", "created_at": "09:11", "priority": "high",   "tenant": "northstar-eu-001"},
    ]

def query_sentry_errors(service: str, since: str) -> list[dict]:
    """READ-ONLY: Query Sentry error tracker."""
    return [
        {"error": "3DSCallbackTimeoutError",     "count": 847, "first_seen": "08:51"},
        {"error": "VATRedirectURLMalformedError", "count": 423, "first_seen": "08:52"},
        {"error": "EnterpriseCheckoutFlowError",  "count": 201, "first_seen": "08:55"},
    ]

# ─── Run evidence gathering ────────────────────────────────────────────────────
print("🔍  Phase 1: Evidence Gathering (Read-Only)")
print("=" * 60)

timeline = EvidenceTimeline(incident_id="INC-2024-EU-001")

# Tool call 1: Datadog
metrics = query_datadog_metrics("checkout-ui", "30m")
timeline.add(EvidenceEntry(
    timestamp="09:04",
    source="Datadog",
    observation=f"Error rate {metrics['error_rate']:.0%}, p99 latency {metrics['p99_latency_ms']}ms",
    inference="Service is severely degraded. Not infrastructure-level (no CPU/memory alerts).",
    confidence="HIGH",
    evidence_id="EV-001",
))

# Tool call 2: Sentry
errors = query_sentry_errors("checkout-ui", "08:45")
for err in errors[:2]:
    timeline.add(EvidenceEntry(
        timestamp=err["first_seen"],
        source="Sentry",
        observation=f"{err['error']} — {err['count']} occurrences",
        inference="3DS and VAT redirect flow is broken. Not a generic server error.",
        confidence="HIGH",
        evidence_id=f"EV-SENTRY-{err['first_seen'].replace(':','')}",
    ))

# Tool call 3: GitHub
deploys = query_github_deployments("checkout-ui")
latest = deploys[0]
timeline.add(EvidenceEntry(
    timestamp=latest["time"][11:16],
    source="GitHub",
    observation=f"Deployment {latest['version']} at {latest['time'][11:16]} — changelog: '{latest['changelog']}'",
    inference="Deployment modified 3DS integration 15 minutes before errors started. High correlation.",
    confidence="HIGH",
    evidence_id="EV-003",
))

# Tool call 4: Zendesk
tickets = search_zendesk_tickets("checkout", "09:00")
timeline.add(EvidenceEntry(
    timestamp="09:02",
    source="Zendesk",
    observation=f"{len(tickets)} urgent tickets — all mention checkout/3DS/VAT issues",
    inference="User-facing impact confirmed. Issue is consistent across multiple tenants.",
    confidence="HIGH",
    evidence_id="EV-004",
))

timeline.gaps = ["Is the 3DS gateway itself degraded, or is this entirely a code bug?"]

timeline.print_summary()
print("\n✅  Evidence gathering complete. Agent stops here — no mutations.")


🔍  Phase 1: Evidence Gathering (Read-Only)
  [Datadog] 09:04 — Error rate 31%, p99 latency 4200ms
    Inference: Service is severely degraded. Not infrastructure-level (no CPU/memory alerts).  (confidence: HIGH)
  [Sentry] 08:51 — 3DSCallbackTimeoutError — 847 occurrences
    Inference: 3DS and VAT redirect flow is broken. Not a generic server error.  (confidence: HIGH)
  [Sentry] 08:52 — VATRedirectURLMalformedError — 423 occurrences
    Inference: 3DS and VAT redirect flow is broken. Not a generic server error.  (confidence: HIGH)
  [GitHub] 08:49 — Deployment v2.1 at 08:49 — changelog: 'Added 3DS v2 support for EU VAT'
    Inference: Deployment modified 3DS integration 15 minutes before errors started. High correlation.  (confidence: HIGH)
  [Zendesk] 09:02 — 4 urgent tickets — all mention checkout/3DS/VAT issues
    Inference: User-facing impact confirmed. Issue is consistent across multiple tenants.  (confidence: HIGH)

  Timeline: 5 evidence entries
  ⚠️  Unresolved gap: Is the 3

---
# Part 2: Impact Synthesis — Blast Radius & SLA Exposure

With evidence gathered, the agent now quantifies the business impact. SLA calculations convert technical failures into financial and legal obligations.

In [ ]:
from pydantic import BaseModel
from typing import Literal
from datetime import datetime, timezone, timedelta

# ─── Tenant database simulation ───────────────────────────────────────────────
TENANT_DB = {
    "northstar-eu-001": {"name": "Northstar Commerce",  "tier": "enterprise", "mrr": 45000, "sla_mins": 15, "region": "EU"},
    "globex-002":       {"name": "Globex Corp",         "tier": "enterprise", "mrr": 38000, "sla_mins": 15, "region": "EU"},
    "acme-003":         {"name": "Acme Ltd",            "tier": "enterprise", "mrr": 22000, "sla_mins": 15, "region": "EU"},
    "umbrella-004":     {"name": "Umbrella Corp",       "tier": "standard",   "mrr": 5000,  "sla_mins": 60, "region": "EU"},
    "initech-005":      {"name": "Initech",             "tier": "standard",   "mrr": 3200,  "sla_mins": 60, "region": "US"},
}

SLA_PENALTY_PCT = 0.10   # 10% of MRR per SLA breach

# ─── Impact models ────────────────────────────────────────────────────────────
class TenantImpact(BaseModel):
    tenant_id: str
    name: str
    tier: str
    mrr: int
    sla_minutes: int
    downtime_minutes: int
    sla_breached: bool
    penalty_usd: float

class IncidentImpact(BaseModel):
    incident_id: str
    total_affected: int
    enterprise_affected: int
    standard_affected: int
    total_mrr_at_risk: int
    total_penalty_usd: float
    tenant_impacts: list[TenantImpact]
    severity: Literal["SEV1", "SEV2", "SEV3"]

def calculate_blast_radius(
    failing_tenant_ids: list[str],
    incident_start: datetime,
    now: datetime,
) -> IncidentImpact:
    """
    Calculate the full business impact of an incident.
    Query tenant DB for tier, SLA, and MRR for each affected tenant.
    """
    downtime_minutes = int((now - incident_start).total_seconds() / 60)
    impacts = []
    
    for tid in failing_tenant_ids:
        tenant = TENANT_DB.get(tid, {})
        sla_mins = tenant.get("sla_mins", 60)
        mrr = tenant.get("mrr", 0)
        breached = downtime_minutes > sla_mins
        penalty = mrr * SLA_PENALTY_PCT if breached else 0
        
        impacts.append(TenantImpact(
            tenant_id=tid,
            name=tenant.get("name", "Unknown"),
            tier=tenant.get("tier", "standard"),
            mrr=mrr,
            sla_minutes=sla_mins,
            downtime_minutes=downtime_minutes,
            sla_breached=breached,
            penalty_usd=penalty,
        ))

    enterprise = [i for i in impacts if i.tier == "enterprise"]
    standard   = [i for i in impacts if i.tier == "standard"]
    total_penalty = sum(i.penalty_usd for i in impacts)
    n_enterprise = len(enterprise)
    
    severity = "SEV1" if (n_enterprise >= 2 or total_penalty > 5000) else (
               "SEV2" if n_enterprise >= 1 else "SEV3")
    
    return IncidentImpact(
        incident_id="INC-2024-EU-001",
        total_affected=len(impacts),
        enterprise_affected=n_enterprise,
        standard_affected=len(standard),
        total_mrr_at_risk=sum(i.mrr for i in impacts),
        total_penalty_usd=total_penalty,
        tenant_impacts=impacts,
        severity=severity,
    )

# ─── Run impact analysis ──────────────────────────────────────────────────────
AFFECTED_TENANTS = ["northstar-eu-001", "globex-002", "acme-003", "umbrella-004"]
start = datetime(2024, 1, 15, 9, 4, tzinfo=timezone.utc)
now   = datetime(2024, 1, 15, 9, 22, tzinfo=timezone.utc)  # 18 minutes in

print("📊  Phase 2: Impact Synthesis")
print("=" * 60)

impact = calculate_blast_radius(AFFECTED_TENANTS, start, now)

print(f"\n  Severity         : {impact.severity}")
print(f"  Downtime         : {impact.tenant_impacts[0].downtime_minutes} minutes")
print(f"  Tenants affected : {impact.total_affected}")
print(f"  Enterprise       : {impact.enterprise_affected} ← priority")
print(f"  MRR at risk      : ${impact.total_mrr_at_risk:,}")
print(f"  SLA penalties    : ${impact.total_penalty_usd:,.0f}")

print(f"\n  Tenant breakdown:")
print(f"  {'Tenant':<20} {'Tier':<12} {'SLA':<8} {'Downtime':<10} {'Breached':<10} {'Penalty'}")
print(f"  {'─'*20} {'─'*12} {'─'*8} {'─'*10} {'─'*10} {'─'*10}")

for t in impact.tenant_impacts:
    breach = "🔴 YES" if t.sla_breached else "🟢 NO"
    print(f"  {t.name:<20} {t.tier:<12} {t.sla_minutes}min    {t.downtime_minutes}min      {breach:<10} ${t.penalty_usd:,.0f}")

print(f"\n⚠️  SLA breach in progress for ALL enterprise accounts (15 min threshold exceeded at {18} min downtime).")


📊  Phase 2: Impact Synthesis

  Severity         : SEV1
  Downtime         : 18 minutes
  Tenants affected : 4
  Enterprise       : 3 ← priority
  MRR at risk      : $110,200
  SLA penalties    : $10,500

  Tenant breakdown:
  Tenant               Tier         SLA      Downtime   Breached   Penalty
  ──────────────────── ──────────── ──────── ────────── ────────── ──────────
  Northstar Commerce   enterprise   15min    18min      🔴 YES     $4,500
  Globex Corp          enterprise   15min    18min      🔴 YES     $3,800
  Acme Ltd             enterprise   15min    18min      🔴 YES     $2,200
  Umbrella Corp        standard     60min    18min      🟢 NO      $0

⚠️  SLA breach in progress for ALL enterprise accounts (15 min threshold exceeded at 18 min downtime).


---
# Part 3: Mitigation Proposals — Idempotent Actions & HITL

The agent **never executes** a production mutation directly. Instead, it produces an `ApprovedMitigationProposal` and routes it to PagerDuty / Slack for human approval. The proposal includes an idempotency key to prevent duplicate execution.

In [ ]:
import hashlib, time
from datetime import datetime, timezone, timedelta
from pydantic import BaseModel
from typing import Literal, Optional

class MitigationProposal(BaseModel):
    incident_id: str
    action: str
    target_service: str
    target_version: str
    justification: str
    evidence_ids: list[str]
    idempotency_key: str
    approval_expires_at: str
    rollback_verification: str
    dry_run_result: str
    status: Literal["PENDING", "APPROVED", "REJECTED", "EXECUTED"] = "PENDING"
    approved_by: Optional[str] = None

def generate_proposal(
    incident_id: str,
    action: str,
    target_service: str,
    target_version: str,
    justification: str,
    evidence_ids: list[str],
) -> MitigationProposal:
    """
    Generate a safe, idempotent proposal. Does NOT execute the action.
    Idempotency key ensures duplicate approvals don't double-execute.
    """
    key_content = f"{incident_id}:{action}:{target_service}:{target_version}"
    idempotency_key = hashlib.sha256(key_content.encode()).hexdigest()[:16]
    
    now = datetime.now(timezone.utc)
    expires = now + timedelta(minutes=30)
    
    # Simulate a dry-run validation
    dry_run_result = (
        f"DRY RUN: Would disable feature flag 'checkout-ui-v2.1' for all tenants. "
        f"Estimated rollback time: 90 seconds. No data mutation required."
    )
    
    return MitigationProposal(
        incident_id=incident_id,
        action=action,
        target_service=target_service,
        target_version=target_version,
        justification=justification,
        evidence_ids=evidence_ids,
        idempotency_key=idempotency_key,
        approval_expires_at=expires.isoformat(),
        rollback_verification="Monitor Datadog error_rate. Success = drops below 2% within 2 minutes.",
        dry_run_result=dry_run_result,
    )

def human_approval_webhook(proposal: MitigationProposal, approved: bool, approver: str) -> MitigationProposal:
    """Simulates the PagerDuty/Slack callback when a human approves."""
    if approved:
        proposal.status = "APPROVED"
        proposal.approved_by = approver
        print(f"  [Webhook] ✅ APPROVED by {approver}")
    else:
        proposal.status = "REJECTED"
        print(f"  [Webhook] ❌ REJECTED by {approver}")
    return proposal

def execute_approved_action(proposal: MitigationProposal) -> str:
    """Only callable after human approval. Uses idempotency_key to prevent duplicates."""
    if proposal.status != "APPROVED":
        raise ValueError(f"Cannot execute — proposal status is '{proposal.status}'")
    
    print(f"  [Orchestrator] 🔐 Executing with idempotency_key={proposal.idempotency_key}")
    print(f"  [Orchestrator] Disabling feature flag 'checkout-ui-v2.1'...")
    time.sleep(0.1)
    proposal.status = "EXECUTED"
    return f"Rollback complete. Feature flag disabled. Idempotency_key={proposal.idempotency_key} marked as used."

# ─── Run proposal workflow ─────────────────────────────────────────────────────
print("🛡️  Phase 3: Mitigation Proposal & Human-in-the-Loop")
print("=" * 60)

proposal = generate_proposal(
    incident_id="INC-2024-EU-001",
    action="disable_feature_flag",
    target_service="checkout-ui",
    target_version="v2.1",
    justification=(
        "Evidence EV-001 through EV-004 consistently link checkout-ui v2.1 "
        "(deployed 08:49) to 3DS callback failures affecting EU enterprise accounts. "
        "3 enterprise tenants in SLA breach. Reverting to v2.0 eliminates the code change."
    ),
    evidence_ids=["EV-001", "EV-SENTRY-0851", "EV-SENTRY-0852", "EV-003", "EV-004"],
)

print(f"\n  Proposal generated:")
print(f"  ┌─────────────────────────────────────────────────────")
print(f"  │ Action          : {proposal.action}")
print(f"  │ Target          : {proposal.target_service} {proposal.target_version}")
print(f"  │ Idempotency Key : {proposal.idempotency_key}")
print(f"  │ Expires At      : {proposal.approval_expires_at[11:19]} UTC")
print(f"  │ Dry-run result  : {proposal.dry_run_result[:60]}...")
print(f"  │ Rollback verify : {proposal.rollback_verification[:60]}...")
print(f"  └─────────────────────────────────────────────────────")
print(f"\n  🛑 AGENT STOPS HERE. Routing proposal to on-call engineer via PagerDuty...")
print(f"\n  ... (simulating 2 minute human review) ...")

# Simulate async human approval
proposal = human_approval_webhook(proposal, approved=True, approver="sarah.chen@northstar.com")

print(f"\n  Executing approved action...")
result = execute_approved_action(proposal)
print(f"  ✅  {result}")
print(f"\n  Monitoring: watching Datadog for error_rate < 2% in next 120 seconds...")


🛡️  Phase 3: Mitigation Proposal & Human-in-the-Loop

  Proposal generated:
  ┌─────────────────────────────────────────────────────
  │ Action          : disable_feature_flag
  │ Target          : checkout-ui v2.1
  │ Idempotency Key : 3a8f7c1e94b20d56
  │ Expires At      : 09:52:00 UTC
  │ Dry-run result  : DRY RUN: Would disable feature flag 'checkout-ui-v2.1'...
  │ Rollback verify : Monitor Datadog error_rate. Success = drops below 2% wit...
  └─────────────────────────────────────────────────────

  🛑 AGENT STOPS HERE. Routing proposal to on-call engineer via PagerDuty...

  ... (simulating 2 minute human review) ...

  [Webhook] ✅ APPROVED by sarah.chen@northstar.com

  Executing approved action...
  [Orchestrator] 🔐 Executing with idempotency_key=3a8f7c1e94b20d56
  [Orchestrator] Disabling feature flag 'checkout-ui-v2.1'...
  ✅  Rollback complete. Feature flag disabled. Idempotency_key=3a8f7c1e94b20d56 marked as used.

  Monitoring: watching Datadog for error_rate < 2% in next 

---
# Capstone Summary: Incident Command Lifecycle

```
09:04  ALERT triggered (Datadog: conversion -31%)
         │
         ▼
       Phase 1: Evidence Gathering (READ-ONLY, ~3 min)
         └── Datadog: error_rate=31%, p99=4200ms
         └── Sentry:  3DSCallbackTimeoutError (847 hits)
         └── GitHub:  checkout-ui v2.1 deployed at 08:49
         └── Zendesk: 4 urgent tickets from EU tenants
         │
         ▼
       Phase 2: Impact Synthesis (~2 min)
         └── 4 tenants affected, 3 enterprise
         └── MRR at risk: $110,200
         └── SLA breach: ALL enterprise (18min > 15min SLA)
         └── Penalty accruing: $10,500
         │
         ▼
       Phase 3: Mitigation Proposal (AGENT STOPS)
         └── Action: disable feature flag 'checkout-ui-v2.1'
         └── Idempotency key: 3a8f7c1e94b20d56
         └── Dry-run: ✅ no data mutation required
         └── Routed to on-call via PagerDuty
         │
         ▼
       Human Approval (09:24 — sarah.chen@northstar.com)
         │
         ▼
       Orchestrator executes (NOT the agent)
         └── Feature flag disabled
         └── Error rate drops: 31% → 0.8% in 90 seconds
09:26  INCIDENT RESOLVED
```

**Production rules:**
1. The agent never executes mutations — it proposes, the orchestrator executes.
2. All proposals are idempotent — clicking "Approve" twice does not double-execute.
3. The approval window expires (30 min) — stale approvals cannot be replayed.
4. Every action is attributed to both the agent (justification) and the human (approver).
